In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

dfs = [pd.read_csv("archive/Dataset/user_" + user + ".csv") for user in ['a', 'b', 'c', 'd']]

for i in range(len(dfs)):
    dfs[i]['User'] = pd.Series(i, index = dfs[i].index)

data = pd.concat(dfs, axis=0).sample(frac=1.0, random_state = 123).reset_index(drop=True)

data = data.drop(['User'], axis=1)
data.head()



,Class,AF3 delta std,AF3 delta m,AF3 theta std,AF3 theta m,AF3 alpha std,AF3 alpha m,AF3 beta std,AF3 beta m,F7 delta std,...,F8 beta std,F8 beta m,AF4 delta std,AF4 delta m,AF4 theta std,AF4 theta m,AF4 alpha std,AF4 alpha m,AF4 beta std,AF4 beta m
0,0.0,3572.252440,2065.056469,0.851824,2.047953,0.651871,2.522036,2.109733,3.347705,3564.779879,...,35.417275,35.151586,3628.426885,2129.789645,5.353671,17.885132,7.672209,29.960618,43.216980,43.932669
1,1.0,3574.116024,2065.528155,1.932513,3.321636,1.138012,2.349805,2.256212,3.945981,3563.399422,...,19.403818,40.800889,3680.341349,2144.200503,10.819521,36.995982,12.812193,24.146774,23.747501,49.072017
2,0.0,3554.487593,2056.215665,0.935015,3.793783,0.736168,2.680542,3.381325,4.678876,3568.839949,...,5.672901,18.176841,3538.347368,2081.315814,5.486555,13.204753,0.664075,6.633072,1.434277,4.132446
3,0.0,3570.668125,2063.974908,1.875394,3.028541,0.910000,3.018672,1.163312,2.883009,3565.403408,...,25.544035,55.547547,3604.601528,2122.493834,15.611283,16.452483,22.462175,46.703612,32.213578,70.892466
4,0.0,3559.747108,2057.401763,1.053691,1.530594,1.593121,2.789907,2.668865,4.326693,3573.651774,...,5.272191,7.208052,3513.244789,2030.461207,1.455450,3.030659,0.482971,3.200647,0.895170,2.099638


In [2]:
# Assuming data is already defined in the previous cell
X = data.drop(columns=['Class']).values
y = data['Class'].values

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert to tensors
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

# Reshape X_tensor to have the correct shape for Conv1d
X_tensor = X_tensor.unsqueeze(1)  # Add a channel dimension

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

class MDCBN(nn.Module):
    def __init__(self, input_channels, num_classes):
        super(MDCBN, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=input_channels, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

        self.lstm = nn.LSTM(128, 64, batch_first=True, bidirectional=True)

        self.fc1 = nn.Linear(64 * 2, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))

        x = x.permute(0, 2, 1)  # Prepare for LSTM
        x, _ = self.lstm(x)
        x = x[:, -1, :]

        x = self.relu(self.bn3(self.fc1(x)))
        x = self.fc2(x)
        return x

# Model setup
input_size = X_tensor.shape[1]
num_classes = 3
model = MDCBN(input_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 100
for epoch in range(epochs):
    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

# Save trained model
torch.save(model.state_dict(), "mdcbn_model.pth")
print("Model training complete and saved as mdcbn_model.pth")

# Calculate accuracy
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in dataloader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs.data, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

accuracy = correct / total
print(f"Model Accuracy: {accuracy:.4f}")


Epoch 1/100, Loss: 1.0646
Epoch 2/100, Loss: 1.0920
Epoch 3/100, Loss: 1.0525
Epoch 4/100, Loss: 1.1222
Epoch 5/100, Loss: 1.0572
Epoch 6/100, Loss: 1.0481
Epoch 7/100, Loss: 0.9844
Epoch 8/100, Loss: 0.9503
Epoch 9/100, Loss: 1.0065
Epoch 10/100, Loss: 0.9687
Epoch 11/100, Loss: 1.0057
Epoch 12/100, Loss: 0.8928
Epoch 13/100, Loss: 0.9426
Epoch 14/100, Loss: 0.9608
Epoch 15/100, Loss: 0.8610
Epoch 16/100, Loss: 0.9881
Epoch 17/100, Loss: 0.8412
Epoch 18/100, Loss: 0.8149
Epoch 19/100, Loss: 0.8905
Epoch 20/100, Loss: 0.7621
Epoch 21/100, Loss: 1.0023
Epoch 22/100, Loss: 0.8117
Epoch 23/100, Loss: 1.1274
Epoch 24/100, Loss: 0.7075
Epoch 25/100, Loss: 0.8568
Epoch 26/100, Loss: 1.1207
Epoch 27/100, Loss: 1.0223
Epoch 28/100, Loss: 0.6136
Epoch 29/100, Loss: 0.5003
Epoch 30/100, Loss: 0.5675
Epoch 31/100, Loss: 0.6164
Epoch 32/100, Loss: 0.7045
Epoch 33/100, Loss: 0.8047
Epoch 34/100, Loss: 0.6811
Epoch 35/100, Loss: 0.5556
Epoch 36/100, Loss: 1.1075
Epoch 37/100, Loss: 0.6948
Epoch 38/1